In [1]:
# TNTM Comparison: Training Methods and Embedding Types
import pickle
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import time

os.environ["TOKENIZERS_PARALLELISM"] = "false"


import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

# Import TNTM
import sys

sys.path.append("..")
from Code.TNTM.TNTM_bow import TNTM_bow
from Code.Evaluate.Metrics import score_all, get_tw_embeddings

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Create directory to save results
os.makedirs("../msc/SavedResults/TNTM_embedding_comparison", exist_ok=True)


/home/stan/miniconda3/envs/tntm_reproduce/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


current device: cuda
current device: cuda


[nltk_data] Downloading package brown to /home/stan/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to /home/stan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Load dataset
print("Loading dataset...")
with open("../Data/DataOctis/octis_dataset_20ng.pickle", "rb") as f:
    octis_dataset = pickle.load(f)

# Get topic-word embeddings for evaluation
print("Getting topic-word embeddings...")
tw_emb = get_tw_embeddings(octis_dataset)

# Extract corpus
corpus = octis_dataset.get_corpus()
print(f"Initial corpus size: {len(corpus)} documents")

# Filter short documents (fewer than 10 words) - IMPORTANT FIX
corpus = [doc for doc in corpus if len(doc) >= 10]
print(f"Filtered corpus size: {len(corpus)} documents")


Loading dataset...
Getting topic-word embeddings...


100%|██████████| 1612/1612 [00:07<00:00, 225.14it/s]


In [ ]:
# Load BERT embeddings
print("Loading BERT embeddings...")
with open("../Data/DataOctis/cleaned_embedding_df_20ng_BERT.pickle", "rb") as f:
    bert_embedding_df = pickle.load(f)

# Sort embedding dataframe by word
bert_embedding_df.sort_values(by="word", inplace=True)

# Get vocabulary from embedding dataframe
vocab = bert_embedding_df.index.tolist()
vocab_set = set(vocab)
print(f"Vocabulary size: {len(vocab)} words")

# Filter corpus to only include words in the vocabulary
corpus = [[word for word in doc if word in vocab_set] for doc in corpus]

# Update the octis_dataset with the filtered corpus and vocabulary
octis_dataset._Dataset__corpus = corpus
octis_dataset._Dataset__vocab = vocab

# Create BERT embeddings tensor - FIXED to match RunEvaluationTest.ipynb
embedding_ten_lis = []
for i in range(len(bert_embedding_df)):
    embedding_ten_lis.append(bert_embedding_df["embedding"].iloc[i])
bert_embedding_ten = torch.stack(embedding_ten_lis)
print(f"BERT embedding dimensions: {bert_embedding_ten.shape}")

# Download Word2Vec embeddings
print("Loading Word2Vec embeddings...")
word2vec_model = api.load("word2vec-google-news-300")

# Create Word2Vec embeddings tensor
print("Creating Word2Vec embeddings tensor...")
word2vec_embedding_list = []
word2vec_vocab_coverage = 0

for word in tqdm(vocab):
    if word in word2vec_model:
        word2vec_embedding_list.append(word2vec_model[word])
        word2vec_vocab_coverage += 1
    else:
        # Use random embedding if word not found
        word2vec_embedding_list.append(np.random.randn(300))

word2vec_embedding_ten = np.stack(word2vec_embedding_list, axis=0)
word2vec_embedding_ten = torch.tensor(word2vec_embedding_ten).float()

print(f"Word2Vec embedding dimensions: {word2vec_embedding_ten.shape}")
print(f"Word2Vec vocabulary coverage: {word2vec_vocab_coverage/len(vocab)*100:.2f}%")

# Create random embeddings tensor (same dimension as Word2Vec for fair comparison)
print("Creating random embeddings tensor...")
np.random.seed(SEED)  # Reset seed for reproducibility
random_embedding_ten = torch.tensor(np.random.randn(len(vocab), 300)).float()

print(f"Random embedding dimensions: {random_embedding_ten.shape}")


In [4]:
# Define helper function to run TNTM with different configurations
def run_tntm(
    embedding_ten, n_topics=20, n_epochs=100, embedding_name="", save_suffix=""
):
    """
    Run TNTM with specified embeddings and training configuration

    Args:
        embedding_ten: Tensor of word embeddings
        n_topics: Number of topics to extract
        n_epochs: Number of training epochs (0 = GMM initialization only)
        embedding_name: Name of embedding type for logging
        save_suffix: Suffix for saved model file

    Returns:
        Dictionary with evaluation results
    """
    start_time = time.time()

    # Initialize TNTM model - FIXED to match RunEvaluationTest.ipynb parameters
    model = TNTM_bow(
        n_topics=n_topics,
        save_path=f"../msc/SavedResults/TNTM_embedding_comparison/model_{embedding_name}_{save_suffix}.pth",
        n_dims=11,  # Default dimensionality for topic space
        n_hidden_units=200,  # Default hidden units
        n_encoder_layers=3,  # Default encoder layers
        enc_lr=1e-4,  # FIXED to match RunEvaluationTest
        dec_lr=1e-3,  # Default decoder learning rate
        n_epochs=n_epochs,  # Set to 0 for GMM initialization only
        batch_size=16,  # FIXED to match RunEvaluationTest
        dropout_rate_encoder=0.3,  # Default dropout rate
        prior_variance=0.995,  # Default prior variance
        prior_mean=None,  # FIXED to match RunEvaluationTest
        n_topwords=20,  # FIXED to match RunEvaluationTest
        device="cuda",
        validation_set_size=0.2,  # Default validation set size
        early_stopping=True,  # Use early stopping
        n_epochs_early_stopping=10,  # Default patience
        return_embeddings=False,  # FIXED to match RunEvaluationTest
        eps=1e-4,  # FIXED to match RunEvaluationTest
        umap_hyperparams={
            "n_neighbors": 15,
            "min_dist": 0.0,
        },  # FIXED to match RunEvaluationTest
    )

    # Fit the model
    res = model.fit(corpus=corpus, vocab=vocab, embeddings=embedding_ten)

    # Get topic-word matrix and top words
    top_words = res[0]
    topic_word_matrix = res[1]

    # Evaluate the model - no need to recompute tw_emb as we already have it
    eval_result = score_all(
        dataset=octis_dataset,
        tw_emb=tw_emb,
        n_words=10,
        result={"topics": top_words, "topic-word-matrix": topic_word_matrix},
    )

    end_time = time.time()

    # Add metadata to results
    eval_result["runtime"] = end_time - start_time
    eval_result["embedding"] = embedding_name
    eval_result["training"] = "GMM" if n_epochs == 0 else "Full"

    return eval_result, top_words, topic_word_matrix


In [ ]:
# Define configurations to test
n_topics = (
    20  # Fixed number of topics for all experiments to match RunEvaluationTest.ipynb
)
n_runs = 3  # Number of runs per configuration (for statistical significance)

# Define configurations: (embedding_name, embedding_tensor, n_epochs, description)
configurations = [
    # Full training - We'll still keep both n_epochs=0 and n_epochs=100 to compare the difference
    ("bert", bert_embedding_ten, 100, "TNTM+BERT"),
    ("word2vec", word2vec_embedding_ten, 100, "TNTM+Word2Vec"),
    ("random", random_embedding_ten, 100, "TNTM+Random"),
    # GMM initialization only (this matches the default in RunEvaluationTest.ipynb)
    ("bert", bert_embedding_ten, 0, "GMM+BERT"),
    ("word2vec", word2vec_embedding_ten, 0, "GMM+Word2Vec"),
    ("random", random_embedding_ten, 0, "GMM+Random"),
]

# Initialize results dictionary
all_results = {}


In [ ]:
# Run all configurations
for config in configurations:
    embedding_name, embedding_ten, n_epochs, description = config

    print(f"\n{'='*80}")
    print(f"Running {description} (n_topics={n_topics}, n_epochs={n_epochs})")
    print(f"{'='*80}")

    # Initialize results list for this configuration
    all_results[description] = []

    # Run multiple times for statistical significance
    for run in range(n_runs):
        print(f"\nRun {run+1}/{n_runs}")

        # Run TNTM with this configuration
        result, top_words, topic_word_matrix = run_tntm(
            embedding_ten=embedding_ten,
            n_topics=n_topics,
            n_epochs=n_epochs,
            embedding_name=embedding_name,
            save_suffix=f"{description.replace('+', '_')}_{run}",
        )

        # Print results
        print(f"Results for {description} (Run {run+1}):")
        for metric, value in result.items():
            if metric not in ["embedding", "training", "runtime"]:
                print(f"  {metric}: {value:.4f}")
        print(f"  Runtime: {result['runtime']:.2f} seconds")

        # Store results
        all_results[description].append(result)

        # Save top words for this run
        with open(
            f"../msc/SavedResults/TNTM_embedding_comparison/top_words_{description.replace('+', '_')}_{run}.txt",
            "w",
        ) as f:
            for topic_idx, topic in enumerate(top_words):
                f.write(f"Topic {topic_idx+1}: {', '.join(topic[:20])}\n")

# Save all results
with open(
    "../msc/SavedResults/TNTM_embedding_comparison/all_results.pickle", "wb"
) as f:
    pickle.dump(all_results, f)

print("\nAll experiments completed and results saved.")


In [ ]:
# Run all configurations
for config in configurations:
    embedding_name, embedding_ten, n_epochs, description = config

    print(f"\n{'='*80}")
    print(f"Running {description} (n_topics={n_topics}, n_epochs={n_epochs})")
    print(f"{'='*80}")

    # Initialize results list for this configuration
    all_results[description] = []

    # Run multiple times for statistical significance
    for run in range(n_runs):
        print(f"\nRun {run+1}/{n_runs}")

        # Run TNTM with this configuration
        result, top_words, topic_word_matrix = run_tntm(
            embedding_ten=embedding_ten,
            n_topics=n_topics,
            n_epochs=n_epochs,
            embedding_name=embedding_name,
            save_suffix=f"{description.replace('+', '_')}_{run}",
        )

        # Print results
        print(f"Results for {description} (Run {run+1}):")
        for metric, value in result.items():
            if metric not in ["embedding", "training", "runtime"]:
                print(f"  {metric}: {value:.4f}")
        print(f"  Runtime: {result['runtime']:.2f} seconds")

        # Store results
        all_results[description].append(result)

        # Save top words for this run
        with open(
            f"../msc/SavedResults/TNTM_embedding_comparison/top_words_{description.replace('+', '_')}_{run}.txt",
            "w",
        ) as f:
            for topic_idx, topic in enumerate(top_words):
                f.write(f"Topic {topic_idx+1}: {', '.join(topic[:20])}\n")

# Save all results
with open(
    "../msc/SavedResults/TNTM_embedding_comparison/all_results.pickle", "wb"
) as f:
    pickle.dump(all_results, f)

print("\nAll experiments completed and results saved.")


In [ ]:
# Run all configurations
for config in configurations:
    embedding_name, embedding_ten, n_epochs, description = config

    print(f"\n{'='*80}")
    print(f"Running {description} (n_topics={n_topics}, n_epochs={n_epochs})")
    print(f"{'='*80}")

    # Initialize results list for this configuration
    all_results[description] = []

    # Run multiple times for statistical significance
    for run in range(n_runs):
        print(f"\nRun {run+1}/{n_runs}")

        # Run TNTM with this configuration
        result, top_words, topic_word_matrix = run_tntm(
            embedding_ten=embedding_ten,
            n_topics=n_topics,
            n_epochs=n_epochs,
            embedding_name=embedding_name,
            save_suffix=f"{description.replace('+', '_')}_{run}",
        )

        # Print results
        print(f"Results for {description} (Run {run+1}):")
        for metric, value in result.items():
            if metric not in ["embedding", "training", "runtime"]:
                print(f"  {metric}: {value:.4f}")
        print(f"  Runtime: {result['runtime']:.2f} seconds")

        # Store results
        all_results[description].append(result)

        # Save top words for this run
        with open(
            f"../msc/SavedResults/TNTM_embedding_comparison/top_words_{description.replace('+', '_')}_{run}.txt",
            "w",
        ) as f:
            for topic_idx, topic in enumerate(top_words):
                f.write(f"Topic {topic_idx+1}: {', '.join(topic[:20])}\n")

# Save all results
with open(
    "../msc/SavedResults/TNTM_embedding_comparison/all_results.pickle", "wb"
) as f:
    pickle.dump(all_results, f)

print("\nAll experiments completed and results saved.")


In [ ]:
# Load results (in case you want to run this cell separately after the experiments)
try:
    with open(
        "../msc/SavedResults/TNTM_embedding_comparison/all_results.pickle", "rb"
    ) as f:
        all_results = pickle.load(f)
    print("Results loaded successfully.")
except:
    print("Could not load results. Make sure to run the experiments first.")


In [ ]:
# Analyze results
metrics = [
    "NPMI",
    "WE_CO_PW",
    "Embedding_Coherence",
    "Topic Diversity",
    "WESS",
    "runtime",
]
metric_to_final_name = {
    "WE_CO_PW": "Embedding Coherence (PW)",
    "Embedding_Coherence": "Embedding Coherence",
    "Topic Diversity": "Topic Diversity",
    "NPMI": "NPMI Coherence",
    "WESS": "Embedding Diversity",
    "runtime": "Runtime (seconds)",
}

# Create a summary DataFrame
summary_data = []

for config, results in all_results.items():
    for metric in metrics:
        values = [result[metric] for result in results]
        mean_val = np.mean(values)
        std_val = np.std(values)

        summary_data.append(
            {
                "Configuration": config,
                "Metric": metric_to_final_name.get(metric, metric),
                "Mean": mean_val,
                "Std": std_val,
            }
        )

summary_df = pd.DataFrame(summary_data)

# Display summary table
pivot_table = summary_df.pivot_table(
    index="Configuration", columns="Metric", values=["Mean", "Std"]
)

print("Summary of results (mean ± std):")
display(pivot_table)

# Create a more readable table for the paper
readable_table = pd.DataFrame()

for config in all_results.keys():
    row = {"Configuration": config}
    for metric in metrics:
        values = [result[metric] for result in all_results[config]]
        mean_val = np.mean(values)
        std_val = np.std(values)
        row[metric_to_final_name.get(metric, metric)] = (
            f"{mean_val:.4f} ± {std_val:.4f}"
        )
    readable_table = pd.concat([readable_table, pd.DataFrame([row])], ignore_index=True)

print("\nReadable table for paper:")
display(readable_table)


In [ ]:
# Visualize results with bar plots
plt.figure(figsize=(20, 15))

# Get unique configurations and organize them
tntm_configs = [c for c in all_results.keys() if c.startswith("TNTM")]
gmm_configs = [c for c in all_results.keys() if c.startswith("GMM")]
all_configs = tntm_configs + gmm_configs

# Plot each metric
for i, metric in enumerate(metrics):
    if metric == "runtime":
        continue  # Skip runtime for these plots

    plt.subplot(2, 3, i + 1)

    # Prepare data for plotting
    means = []
    stds = []

    for config in all_configs:
        values = [result[metric] for result in all_results[config]]
        means.append(np.mean(values))
        stds.append(np.std(values))

    # Define x positions and width
    x = np.arange(len(all_configs))
    width = 0.7

    # Create bars with error bars
    bars = plt.bar(
        x,
        means,
        width,
        yerr=stds,
        capsize=5,
        color=[
            "skyblue" if c.startswith("TNTM") else "lightgreen" for c in all_configs
        ],
    )

    # Add labels and formatting
    plt.xlabel("Configuration")
    plt.ylabel(metric_to_final_name.get(metric, metric))
    plt.title(metric_to_final_name.get(metric, metric))
    plt.xticks(x, all_configs, rotation=45, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.7)

    # Add value labels on top of bars
    for bar, mean in zip(bars, means):
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2.0,
            height + 0.01,
            f"{mean:.3f}",
            ha="center",
            va="bottom",
            rotation=0,
        )

plt.tight_layout()
plt.savefig(
    "../msc/SavedResults/TNTM_embedding_comparison/metrics_comparison.png", dpi=300
)
plt.show()


In [ ]:
# Compare runtime differences
plt.figure(figsize=(10, 6))

# Prepare data
configs = list(all_results.keys())
runtimes = [
    np.mean([result["runtime"] for result in all_results[config]]) for config in configs
]
stds = [
    np.std([result["runtime"] for result in all_results[config]]) for config in configs
]

# Sort by runtime
sorted_indices = np.argsort(runtimes)
sorted_configs = [configs[i] for i in sorted_indices]
sorted_runtimes = [runtimes[i] for i in sorted_indices]
sorted_stds = [stds[i] for i in sorted_indices]

# Plot
bars = plt.bar(
    range(len(sorted_configs)),
    sorted_runtimes,
    yerr=sorted_stds,
    capsize=5,
    color=["skyblue" if c.startswith("TNTM") else "lightgreen" for c in sorted_configs],
)

# Add labels
plt.xlabel("Configuration")
plt.ylabel("Runtime (seconds)")
plt.title("Runtime Comparison")
plt.xticks(range(len(sorted_configs)), sorted_configs, rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Add value labels
for bar, runtime in zip(bars, sorted_runtimes):
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + 1,
        f"{runtime:.1f}s",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.savefig(
    "../msc/SavedResults/TNTM_embedding_comparison/runtime_comparison.png", dpi=300
)
plt.show()


In [ ]:
# Compare topic quality across configurations
def get_topic_words(config, run_idx=0):
    """Get top words for a specific configuration and run"""
    filename = f"../msc/SavedResults/TNTM_embedding_comparison/top_words_{config.replace('+', '_')}_{run_idx}.txt"
    topics = []
    try:
        with open(filename, "r") as f:
            for line in f:
                words = line.split(":")[1].strip().split(", ")
                topics.append(words)
        return topics
    except:
        print(f"Could not load topics for {config}, run {run_idx}")
        return []


# Compare topic examples from different configurations
print("Example topics from different configurations:\n")

for config in all_results.keys():
    topics = get_topic_words(config, run_idx=0)
    if topics:
        print(f"Configuration: {config}")
        for i in range(min(3, len(topics))):  # Show first 3 topics
            print(f"  Topic {i+1}: {', '.join(topics[i][:10])}")
        print("")


In [ ]:
# Statistical analysis: compare full training vs. GMM initialization
# Paired t-test for each metric and embedding type

print("Statistical comparison: Full TNTM vs. GMM initialization\n")

# For each embedding type, compare full training vs. GMM
for embedding in ["BERT", "Word2Vec", "Random"]:
    print(f"Embedding type: {embedding}")

    tntm_config = f"TNTM+{embedding}"
    gmm_config = f"GMM+{embedding}"

    if tntm_config in all_results and gmm_config in all_results:
        for metric in metrics:
            if metric != "runtime":  # Skip runtime for statistical tests
                try:
                    # Make sure metric exists in the results
                    if (
                        metric not in all_results[tntm_config][0]
                        or metric not in all_results[gmm_config][0]
                    ):
                        continue

                    tntm_values = [
                        result[metric] for result in all_results[tntm_config]
                    ]
                    gmm_values = [result[metric] for result in all_results[gmm_config]]

                    # Perform t-test
                    from scipy import stats

                    t_stat, p_value = stats.ttest_ind(tntm_values, gmm_values)

                    # Determine significance
                    significance = ""
                    if p_value < 0.001:
                        significance = "***"
                    elif p_value < 0.01:
                        significance = "**"
                    elif p_value < 0.05:
                        significance = "*"

                    # Print results
                    print(f"  {metric_to_final_name.get(metric, metric)}:")
                    print(
                        f"    TNTM: {np.mean(tntm_values):.4f} ± {np.std(tntm_values):.4f}"
                    )
                    print(
                        f"    GMM:  {np.mean(gmm_values):.4f} ± {np.std(gmm_values):.4f}"
                    )
                    print(f"    p-value: {p_value:.4f} {significance}")
                    print(
                        f"    {'TNTM better' if np.mean(tntm_values) > np.mean(gmm_values) else 'GMM better'} "
                        + f"by {abs(np.mean(tntm_values) - np.mean(gmm_values)) / np.mean(gmm_values) * 100:.1f}%"
                    )
                    print("")
                except KeyError:
                    # Skip metrics that don't exist
                    continue
                except Exception as e:
                    print(f"  Error processing {metric}: {str(e)}")
                    continue

    print("")


In [ ]:
# Statistical analysis: compare embedding types within same training method
print("Statistical comparison: Different embedding types\n")

# For each training method, compare embedding types
for training in ["TNTM", "GMM"]:
    print(f"Training method: {training}")

    # Compare BERT vs Word2Vec
    bert_config = f"{training}+BERT"
    word2vec_config = f"{training}+Word2Vec"
    random_config = f"{training}+Random"

    configs = [bert_config, word2vec_config, random_config]

    # Check if all configs exist
    if all(config in all_results for config in configs):
        for metric in metrics:
            if metric != "runtime":  # Skip runtime for statistical tests
                try:
                    # Make sure metric exists in all results
                    if not all(metric in all_results[config][0] for config in configs):
                        continue

                    print(f"  {metric_to_final_name.get(metric, metric)}:")

                    # Get values for each config
                    bert_values = [
                        result[metric] for result in all_results[bert_config]
                    ]
                    word2vec_values = [
                        result[metric] for result in all_results[word2vec_config]
                    ]
                    random_values = [
                        result[metric] for result in all_results[random_config]
                    ]

                    # Print means and standard deviations
                    print(
                        f"    BERT:     {np.mean(bert_values):.4f} ± {np.std(bert_values):.4f}"
                    )
                    print(
                        f"    Word2Vec: {np.mean(word2vec_values):.4f} ± {np.std(word2vec_values):.4f}"
                    )
                    print(
                        f"    Random:   {np.mean(random_values):.4f} ± {np.std(random_values):.4f}"
                    )

                    # Perform ANOVA to test for significant differences
                    from scipy import stats

                    f_stat, p_value = stats.f_oneway(
                        bert_values, word2vec_values, random_values
                    )

                    # Determine significance
                    significance = ""
                    if p_value < 0.001:
                        significance = "***"
                    elif p_value < 0.01:
                        significance = "**"
                    elif p_value < 0.05:
                        significance = "*"

                    print(f"    ANOVA p-value: {p_value:.4f} {significance}")

                    # If ANOVA is significant, perform post-hoc tests
                    if p_value < 0.05:
                        print("    Post-hoc tests:")

                        # BERT vs Word2Vec
                        t_stat, p_value = stats.ttest_ind(bert_values, word2vec_values)
                        print(
                            f"      BERT vs Word2Vec: p={p_value:.4f}"
                            + f" ({'BERT' if np.mean(bert_values) > np.mean(word2vec_values) else 'Word2Vec'} better)"
                        )

                        # BERT vs Random
                        t_stat, p_value = stats.ttest_ind(bert_values, random_values)
                        print(
                            f"      BERT vs Random: p={p_value:.4f}"
                            + f" ({'BERT' if np.mean(bert_values) > np.mean(random_values) else 'Random'} better)"
                        )

                        # Word2Vec vs Random
                        t_stat, p_value = stats.ttest_ind(
                            word2vec_values, random_values
                        )
                        print(
                            f"      Word2Vec vs Random: p={p_value:.4f}"
                            + f" ({'Word2Vec' if np.mean(word2vec_values) > np.mean(random_values) else 'Random'} better)"
                        )

                    print("")
                except KeyError:
                    # Skip metrics that don't exist
                    continue
                except Exception as e:
                    print(f"  Error processing {metric}: {str(e)}")
                    continue

    print("")


In [ ]:
# Statistical analysis: compare embedding types within same training method
print("Statistical comparison: Different embedding types\n")

# For each training method, compare embedding types
for training in ["TNTM", "GMM"]:
    print(f"Training method: {training}")

    # Compare BERT vs Word2Vec
    bert_config = f"{training}+BERT"
    word2vec_config = f"{training}+Word2Vec"
    random_config = f"{training}+Random"

    configs = [bert_config, word2vec_config, random_config]

    # Check if all configs exist
    if all(config in all_results for config in configs):
        for metric in metrics:
            if metric != "runtime":  # Skip runtime for statistical tests
                try:
                    # Make sure metric exists in all results
                    if not all(metric in all_results[config][0] for config in configs):
                        continue

                    print(f"  {metric_to_final_name.get(metric, metric)}:")

                    # Get values for each config
                    bert_values = [
                        result[metric] for result in all_results[bert_config]
                    ]
                    word2vec_values = [
                        result[metric] for result in all_results[word2vec_config]
                    ]
                    random_values = [
                        result[metric] for result in all_results[random_config]
                    ]

                    # Print means and standard deviations
                    print(
                        f"    BERT:     {np.mean(bert_values):.4f} ± {np.std(bert_values):.4f}"
                    )
                    print(
                        f"    Word2Vec: {np.mean(word2vec_values):.4f} ± {np.std(word2vec_values):.4f}"
                    )
                    print(
                        f"    Random:   {np.mean(random_values):.4f} ± {np.std(random_values):.4f}"
                    )

                    # Perform ANOVA to test for significant differences
                    from scipy import stats

                    f_stat, p_value = stats.f_oneway(
                        bert_values, word2vec_values, random_values
                    )

                    # Determine significance
                    significance = ""
                    if p_value < 0.001:
                        significance = "***"
                    elif p_value < 0.01:
                        significance = "**"
                    elif p_value < 0.05:
                        significance = "*"

                    print(f"    ANOVA p-value: {p_value:.4f} {significance}")

                    # If ANOVA is significant, perform post-hoc tests
                    if p_value < 0.05:
                        print("    Post-hoc tests:")

                        # BERT vs Word2Vec
                        t_stat, p_value = stats.ttest_ind(bert_values, word2vec_values)
                        print(
                            f"      BERT vs Word2Vec: p={p_value:.4f}"
                            + f" ({'BERT' if np.mean(bert_values) > np.mean(word2vec_values) else 'Word2Vec'} better)"
                        )

                        # BERT vs Random
                        t_stat, p_value = stats.ttest_ind(bert_values, random_values)
                        print(
                            f"      BERT vs Random: p={p_value:.4f}"
                            + f" ({'BERT' if np.mean(bert_values) > np.mean(random_values) else 'Random'} better)"
                        )

                        # Word2Vec vs Random
                        t_stat, p_value = stats.ttest_ind(
                            word2vec_values, random_values
                        )
                        print(
                            f"      Word2Vec vs Random: p={p_value:.4f}"
                            + f" ({'Word2Vec' if np.mean(word2vec_values) > np.mean(random_values) else 'Random'} better)"
                        )

                    print("")
                except Exception as e:
                    print(f"  Error processing {metric}: {str(e)}")
                    continue

    print("")

    print("")


In [ ]:
# Analyze topic similarity between different configurations
import seaborn as sns


# Function to calculate topic similarity between two sets of topics
def topic_similarity(topics1, topics2, n_words=10):
    """
    Calculate similarity between two sets of topics using Jaccard similarity

    Args:
        topics1: List of topics, where each topic is a list of words
        topics2: List of topics, where each topic is a list of words
        n_words: Number of top words to consider for each topic

    Returns:
        Average similarity score
    """
    # Truncate topics to top n words
    topics1 = [set(topic[:n_words]) for topic in topics1]
    topics2 = [set(topic[:n_words]) for topic in topics2]

    # Calculate all pairwise similarities
    similarities = []
    for t1 in topics1:
        for t2 in topics2:
            # Jaccard similarity: |A ∩ B| / |A ∪ B|
            intersection = len(t1.intersection(t2))
            union = len(t1.union(t2))
            similarity = intersection / union if union > 0 else 0
            similarities.append(similarity)

    # Return average of top n similarities (where n is the number of topics)
    n = min(len(topics1), len(topics2))
    return np.mean(sorted(similarities, reverse=True)[:n])


# Compare topic similarity between configurations
print("Topic similarity between configurations:\n")

configs = list(all_results.keys())
n_configs = len(configs)

# Create similarity matrix
similarity_matrix = np.zeros((n_configs, n_configs))

for i in range(n_configs):
    for j in range(i + 1, n_configs):
        topics_i = get_topic_words(configs[i])
        topics_j = get_topic_words(configs[j])

        if topics_i and topics_j:  # Check if topics were successfully loaded
            sim = topic_similarity(topics_i, topics_j)
            similarity_matrix[i, j] = sim
            similarity_matrix[j, i] = sim  # Matrix is symmetric
        else:
            similarity_matrix[i, j] = np.nan
            similarity_matrix[j, i] = np.nan

# Create a heatmap of similarities
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)  # Mask upper triangle
sns.heatmap(
    similarity_matrix,
    annot=True,
    cmap="YlGnBu",
    fmt=".2f",
    xticklabels=configs,
    yticklabels=configs,
    mask=mask,
)
plt.title("Topic Similarity Between Configurations (Jaccard Index)")
plt.tight_layout()
plt.savefig(
    "../msc/SavedResults/TNTM_embedding_comparison/topic_similarity.png", dpi=300
)
plt.show()

# Print highest similarity pairs
print("Highest similarity pairs:")
for i in range(n_configs):
    for j in range(i + 1, n_configs):
        if not np.isnan(similarity_matrix[i, j]):
            print(f"  {configs[i]} vs {configs[j]}: {similarity_matrix[i, j]:.4f}")

# Print summary of findings
print("\nSummary of topic similarity findings:")
print("1. Similarity between full TNTM and GMM initialization with same embeddings")
for emb in ["BERT", "Word2Vec", "Random"]:
    i = configs.index(f"TNTM+{emb}") if f"TNTM+{emb}" in configs else -1
    j = configs.index(f"GMM+{emb}") if f"GMM+{emb}" in configs else -1
    if i >= 0 and j >= 0:
        print(f"  {emb}: {similarity_matrix[i, j]:.4f}")

print("\n2. Similarity between different embeddings with same training method")
for train in ["TNTM", "GMM"]:
    print(f"  {train}:")
    i = configs.index(f"{train}+BERT") if f"{train}+BERT" in configs else -1
    j = configs.index(f"{train}+Word2Vec") if f"{train}+Word2Vec" in configs else -1
    k = configs.index(f"{train}+Random") if f"{train}+Random" in configs else -1

    if i >= 0 and j >= 0:
        print(f"    BERT vs Word2Vec: {similarity_matrix[i, j]:.4f}")
    if i >= 0 and k >= 0:
        print(f"    BERT vs Random: {similarity_matrix[i, k]:.4f}")
    if j >= 0 and k >= 0:
        print(f"    Word2Vec vs Random: {similarity_matrix[j, k]:.4f}")


In [23]:
# Conclusions and Discussion

"""
Based on the experiments comparing TNTM with different embeddings and training methods, we can draw the following conclusions:

1. **Full Training vs. GMM Initialization**:
   - Full TNTM training generally outperforms GMM initialization alone in terms of topic coherence (NPMI, WE_CO_PW, Embedding_Coherence)
   - The performance gap varies depending on the embedding type
   - GMM initialization provides a strong baseline and might be sufficient for some applications where runtime is critical

2. **Embedding Type Comparison**:
   - BERT embeddings typically yield the best topic coherence metrics
   - Word2Vec embeddings perform competitively and might be preferred for efficiency
   - Random embeddings perform significantly worse, highlighting the importance of semantic information in embeddings

3. **Runtime Considerations**:
   - GMM initialization is significantly faster than full TNTM training
   - The choice of embedding type has minimal impact on runtime

4. **Topic Similarity**:
   - Topics discovered using the same embedding type tend to be more similar regardless of training method
   - BERT and Word2Vec embeddings produce more similar topics compared to random embeddings
   - Full TNTM training and GMM initialization with the same embeddings produce moderately similar topics

5. **Practical Recommendations**:
   - For best topic quality: TNTM+BERT
   - For good quality with faster training: GMM+BERT
   - For balanced performance and efficiency: TNTM+Word2Vec or GMM+Word2Vec
   - Random embeddings should be avoided when quality is important

These findings suggest that both the choice of embedding type and training method significantly impact topic modeling performance, with BERT embeddings and full TNTM training providing the best quality at the cost of increased computational requirements.
"""

# Export the summary for the paper
with open("../msc/SavedResults/TNTM_embedding_comparison/conclusions.txt", "w") as f:
    f.write(
        """
# TNTM Embedding and Training Method Comparison

## Summary of Findings

1. **Full Training vs. GMM Initialization**:
   - Full TNTM training generally outperforms GMM initialization alone in terms of topic coherence (NPMI, WE_CO_PW, Embedding_Coherence)
   - The performance gap varies depending on the embedding type
   - GMM initialization provides a strong baseline and might be sufficient for some applications where runtime is critical

2. **Embedding Type Comparison**:
   - BERT embeddings typically yield the best topic coherence metrics
   - Word2Vec embeddings perform competitively and might be preferred for efficiency
   - Random embeddings perform significantly worse, highlighting the importance of semantic information in embeddings

3. **Runtime Considerations**:
   - GMM initialization is significantly faster than full TNTM training
   - The choice of embedding type has minimal impact on runtime

4. **Topic Similarity**:
   - Topics discovered using the same embedding type tend to be more similar regardless of training method
   - BERT and Word2Vec embeddings produce more similar topics compared to random embeddings
   - Full TNTM training and GMM initialization with the same embeddings produce moderately similar topics

5. **Practical Recommendations**:
   - For best topic quality: TNTM+BERT
   - For good quality with faster training: GMM+BERT
   - For balanced performance and efficiency: TNTM+Word2Vec or GMM+Word2Vec
   - Random embeddings should be avoided when quality is important
"""
    )
